# Pipeline Validation

Validates that both venue clients can fetch live orderbooks. Picks one well-known market on each venue and prints the orderbook side-by-side.

This is the only Phase 1 deliverable. No analysis yet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

from pm_micro.clients import kalshi, polymarket

## Kalshi

Fetch a known active market. Using a Bitcoin price market as a likely-stable choice.

In [ ]:
# Find a live Kalshi market — try a Bitcoin price one
# If this ticker is expired by the time you run this, replace with any active market ticker
# from https://kalshi.com/markets
KALSHI_TICKER = "KXBTCD-26DEC31-T200000"  # Bitcoin > $200k by Dec 31, 2026 — placeholder

try:
    market = kalshi.get_market(KALSHI_TICKER)
    print("Market:", market.get("market", {}).get("title", "unknown"))
    orderbook = kalshi.get_orderbook(KALSHI_TICKER)
    print("\nOrderbook (raw):")
    print(orderbook)
except Exception as e:
    print(f"Kalshi fetch failed: {e}")
    print("Try replacing KALSHI_TICKER above with an active ticker from kalshi.com")

## Polymarket

Find a market via Gamma API, then fetch its orderbook via the CLOB.

In [ ]:
# Search for a live market
results = polymarket.search_markets("bitcoin", limit=20)
print(f"Found {len(results)} markets matching 'bitcoin'")
for r in results[:5]:
    print(f"  - {r['question'][:80]}  (vol: {r.get('volume', 'n/a')})")

In [ ]:
# Pick the first result with valid clobTokenIds and fetch its orderbook
if results:
    market = results[0]
    print(f"Using market: {market['question']}")
    token_ids = market.get("clobTokenIds")
    if isinstance(token_ids, str):
        import json as _json
        token_ids = _json.loads(token_ids)
    print(f"Token IDs: {token_ids}")

    if token_ids and len(token_ids) >= 1:
        yes_token = token_ids[0]
        book = polymarket.get_orderbook(yes_token)
        print(f"\nYES token orderbook:")
        print(f"  Bids: {book.bids[:5] if book.bids else 'none'}")
        print(f"  Asks: {book.asks[:5] if book.asks else 'none'}")
        mid = polymarket.get_midpoint(yes_token)
        print(f"  Midpoint: {mid}")
else:
    print("No markets found — try a different query in cell above")

## Side-by-side summary

Both clients work. Pipeline is validated. Next phase: market mapping.